In [8]:
import polars as pl
from rdkit import Chem
import time

def canonicalize(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return (None, None)
        canon = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        return (canon)
    except Exception:
        return (None, None)

def main():
    df = pl.read_parquet("20250826_135514.parquet").select("smiles").head(100_000)  # sample 100k
    smiles_list = df["smiles"].to_list()

    t0 = time.time()
    results = [canonicalize(s) for s in smiles_list]
    elapsed = time.time() - t0

    canon, ikeys = zip(*results)
    df = df.with_columns([
        pl.Series("canonical_smiles", canon),
    ])

    print(f"Processed {len(smiles_list)} molecules in {elapsed:.2f} seconds")
    print(f"Throughput: {len(smiles_list)/elapsed:.2f} molecules/second")

main()

[08:00:27] SMILES Parse Error: syntax error while parsing: C([C@H](C)CO)C{-}/C=C(\C{+n}C/C=C(/CC/C=C(\C)/CC/C=C(\C)/C)\C)/C
[08:00:27] SMILES Parse Error: check for mistakes around position 15:
[08:00:27] C([C@H](C)CO)C{-}/C=C(\C{+n}C/C=C(/CC/C=C
[08:00:27] ~~~~~~~~~~~~~~^
[08:00:27] SMILES Parse Error: Failed parsing SMILES 'C([C@H](C)CO)C{-}/C=C(\C{+n}C/C=C(/CC/C=C(\C)/CC/C=C(\C)/C)\C)/C' for input: 'C([C@H](C)CO)C{-}/C=C(\C{+n}C/C=C(/CC/C=C(\C)/CC/C=C(\C)/C)\C)/C'
[08:00:27] SMILES Parse Error: syntax error while parsing: C([C@H](C)C(=O)O)C{-}/C=C(\C{+n}C/C=C(/CC/C=C(\C)/CC/C=C(\C)/C)\C)/C
[08:00:27] SMILES Parse Error: check for mistakes around position 19:
[08:00:27] C([C@H](C)C(=O)O)C{-}/C=C(\C{+n}C/C=C(/CC
[08:00:27] ~~~~~~~~~~~~~~~~~~^
[08:00:27] SMILES Parse Error: Failed parsing SMILES 'C([C@H](C)C(=O)O)C{-}/C=C(\C{+n}C/C=C(/CC/C=C(\C)/CC/C=C(\C)/C)\C)/C' for input: 'C([C@H](C)C(=O)O)C{-}/C=C(\C{+n}C/C=C(/CC/C=C(\C)/CC/C=C(\C)/C)\C)/C'
[08:00:27] SMILES Parse Error: syntax er

Processed 49267 molecules in 8.79 seconds
Throughput: 5602.75 molecules/second


In [9]:
import psycopg2
import time
from psycopg2.extras import RealDictCursor

# PostgreSQL connection parameters
conn_params = {
    "host": "localhost",
    "database": "metabo",
    "user": "postgres",
    "password": "postgres",
    "port": 5436
}

def test_postgres_rdkit():
    """Test RDKit canonicalization in PostgreSQL on silver.compounds table"""
    
    conn = psycopg2.connect(**conn_params)
    cur = conn.cursor()
    
    try:
        # First, let's check how many rows we have
        cur.execute("SELECT COUNT(*) FROM silver.compounds WHERE smiles IS NOT NULL")
        total_count = cur.fetchone()[0]
        print(f"Total compounds with SMILES: {total_count}")
        
        # Test canonicalization performance on a sample
        sample_size = 1000000
        print(f"\nTesting canonicalization on {sample_size} compounds...")
        
        # Using RDKit PostgreSQL extension functions
        query = """
        WITH sample AS (
            SELECT smiles 
            FROM silver.compounds 
            WHERE smiles IS NOT NULL 
            LIMIT %s
        )
        SELECT 
            smiles,
            mol_to_smiles(mol_from_smiles(smiles)) as canonical_smiles
        FROM sample
        """
        
        t0 = time.time()
        cur.execute(query, (sample_size,))
        results = cur.fetchall()
        elapsed = time.time() - t0
        
        # Count successful canonicalizations
        successful = sum(1 for r in results if r[1] is not None)
        
        print(f"Processed {len(results)} molecules in {elapsed:.2f} seconds")
        print(f"Throughput: {len(results)/elapsed:.2f} molecules/second")
        print(f"Successfully canonicalized: {successful}/{len(results)} ({successful/len(results)*100:.1f}%)")
        
        # Show a few examples
        print("\nFirst 5 examples:")
        for i, (original, canonical) in enumerate(results[:5]):
            print(f"  {i+1}. Original: {original[:50]}...")
            print(f"     Canonical: {canonical[:50] if canonical else 'FAILED'}...")
        
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()
    finally:
        cur.close()
        conn.close()

test_postgres_rdkit()

Total compounds with SMILES: 1329632

Testing canonicalization on 1000000 compounds...
Processed 1000000 molecules in 318.56 seconds
Throughput: 3139.12 molecules/second
Successfully canonicalized: 999992/1000000 (100.0%)

First 5 examples:
  1. Original: OCC(O)=O...
     Canonical: O=C(O)CO...
  2. Original: OCC(O)=O...
     Canonical: O=C(O)CO...
  3. Original: OCC(O)C(=O)C(O)CO...
     Canonical: O=C(C(O)CO)C(O)CO...
  4. Original: OCC(O)C(=O)P1(=O)C2(O)C(O)C(O)C(O)C(O)C12O...
     Canonical: O=C(C(O)CO)P1(=O)C2(O)C(O)C(O)C(O)C(O)C21O...
  5. Original: OCC(O)C(CO)OCN1C=CN=C1[N+]([O-])=O...
     Canonical: O=[N+]([O-])c1nccn1COC(CO)C(O)CO...
